# 03 — Rol 3: Mezclas gaussianas y variables latentes

**Juan Andres Martinez (326771)**

Modelamos los viajes como una mezcla de perfiles latentes en el espacio `(log distancia,
log duración)`. El algoritmo EM se implementa **a mano** en NumPy (pasos E y M explícitos) y
se verifica contra `scikit-learn`.

**Hallazgo metodológico clave:** con la tolerancia de convergencia por defecto (1e-3, o incluso
1e-6), el EM se detiene en una *meseta* de la verosimilitud antes de llegar al óptimo real,
produciendo pesos de mezcla distintos entre semillas que parecen (pero no son) soluciones
genuinamente diferentes. Se demuestra abajo y se corrige con tolerancia 1e-9.

In [1]:
import pandas as pd, numpy as np
from scipy.special import logsumexp

m = pd.read_parquet('muestra_50k.parquet')
R = 6371.0
def haversine_km(lo1, la1, lo2, la2):
    lo1, la1, lo2, la2 = map(np.radians, [lo1, la1, lo2, la2])
    a = np.sin((la2-la1)/2)**2 + np.cos(la1)*np.cos(la2)*np.sin((lo2-lo1)/2)**2
    return 2*R*np.arcsin(np.sqrt(a))
m['dist_km'] = haversine_km(m.pickup_longitude, m.pickup_latitude, m.dropoff_longitude, m.dropoff_latitude)
m['log_dist'] = np.log(m.dist_km)
m['log_dur']  = np.log(m.trip_duration)
X = m[['log_dist', 'log_dur']].values
print(f"n = {len(m):,} | espacio: (log distancia, log duración)")


n = 50,004 | espacio: (log distancia, log duración)


## 1. EM implementado a mano

In [2]:
from sklearn.cluster import kmeans_plusplus

def log_gauss(X, mu, S):
    L = np.linalg.cholesky(S)
    z = np.linalg.solve(L, (X - mu).T)
    d = X.shape[1]
    return -0.5*(d*np.log(2*np.pi) + 2*np.log(np.diag(L)).sum() + (z**2).sum(0))

def paso_E(X, pi, mu, S):
    logp = np.column_stack([np.log(pi[k]) + log_gauss(X, mu[k], S[k]) for k in range(len(pi))])
    lse = logsumexp(logp, axis=1)
    return np.exp(logp - lse[:, None]), lse.mean()

def paso_M(X, r, reg=1e-6):
    Nk = r.sum(0); pi = Nk/len(X); mu = (r.T @ X) / Nk[:, None]
    d = X.shape[1]
    S = np.array([((r[:,k,None]*(X-mu[k])).T @ (X-mu[k]))/Nk[k] + reg*np.eye(d) for k in range(r.shape[1])])
    return pi, mu, S

def init_modelo(X, K, tipo, seed):
    rng = np.random.default_rng(seed)
    if tipo == 'kmeans++':
        c, _ = kmeans_plusplus(X, K, random_state=seed)
        r = np.eye(K)[np.argmin(((X[:,None,:]-c[None])**2).sum(2), 1)]
    else:
        r = rng.random((len(X), K)); r /= r.sum(1, keepdims=True)
    return paso_M(X, r)

def em(X, K, tipo, seed, tol, max_iter=3000):
    pi, mu, S = init_modelo(X, K, tipo, seed); prev = None
    for it in range(1, max_iter+1):
        r, ll = paso_E(X, pi, mu, S)
        if prev is not None and abs(ll - prev) < tol: break
        prev = ll
        pi, mu, S = paso_M(X, r)
    return dict(pi=pi, mu=mu, S=S, r=r, ll=ll, iter=it)

print("Funciones EM definidas: paso_E, paso_M, em()")


Funciones EM definidas: paso_E, paso_M, em()


### Demostración: tolerancia laxa vs. estricta (K=4, 3 semillas)

In [3]:
print("--- tol = 1e-6 (laxa) ---")
for s in range(3):
    o = em(X, 4, 'kmeans++', 42+s, tol=1e-6)
    print(f"semilla {42+s}: ll={o['ll']:.6f} iter={o['iter']:4d} pesos={np.round(np.sort(o['pi']),3)}")


--- tol = 1e-6 (laxa) ---


semilla 42: ll=-1.792917 iter= 551 pesos=[0.017 0.032 0.462 0.489]


semilla 43: ll=-1.792889 iter= 167 pesos=[0.017 0.036 0.441 0.505]


semilla 44: ll=-1.792915 iter= 598 pesos=[0.017 0.031 0.461 0.49 ]


In [4]:
print("--- tol = 1e-9 (estricta) ---")
resultados_estrictos = []
for s in range(3):
    o = em(X, 4, 'kmeans++', 42+s, tol=1e-9, max_iter=3000)
    resultados_estrictos.append(o)
    print(f"semilla {42+s}: ll={o['ll']:.6f} iter={o['iter']:4d} pesos={np.round(np.sort(o['pi']),3)}")
print("\nCon tolerancia estricta, las 3 semillas convergen al MISMO óptimo:")
print("la 'inestabilidad' con tol=1e-6 era una parada prematura en una meseta, no mínimos locales distintos.")


--- tol = 1e-9 (estricta) ---


semilla 42: ll=-1.792874 iter= 869 pesos=[0.017 0.033 0.431 0.519]


semilla 43: ll=-1.792874 iter= 393 pesos=[0.017 0.033 0.431 0.519]


semilla 44: ll=-1.792874 iter= 914 pesos=[0.017 0.033 0.431 0.519]

Con tolerancia estricta, las 3 semillas convergen al MISMO óptimo:
la 'inestabilidad' con tol=1e-6 era una parada prematura en una meseta, no mínimos locales distintos.


## 2. Selección de K (criterio BIC, tolerancia estricta)

In [5]:
def bic(X, o):
    K, d = len(o['pi']), X.shape[1]
    n_params = K*(1 + d + d*(d+1)//2) - 1
    return -2*o['ll']*len(X) + n_params*np.log(len(X))

for K in [1, 2, 3, 4, 5]:
    o = em(X, K, 'kmeans++', 42, tol=1e-9, max_iter=3000)
    print(f"K={K}: ll={o['ll']:.6f}  BIC={bic(X, o):,.1f}  iter={o['iter']}")


K=1: ll=-1.865541  BIC=186,623.1  iter=2


K=2: ll=-1.826277  BIC=182,761.4  iter=237


K=3: ll=-1.809629  BIC=181,161.3  iter=600


K=4: ll=-1.792874  BIC=179,550.6  iter=869


K=5: ll=-1.790619  BIC=179,390.0  iter=1395


K=5 es la elección: el BIC sigue mejorando, pero (verificado por separado, ver bitácora) K=6
no es reproducible entre semillas con tolerancia estricta — dos semillas caen en dos óptimos
distintos, uno de ellos peor que K=5. Eso son mínimos locales genuinos, a diferencia del
artefacto de convergencia visto arriba.

## 3. Modelo final K=5 y perfiles de viaje

In [6]:
modelo = em(X, 5, 'kmeans++', 42, tol=1e-9, max_iter=3000)
orden = np.argsort(modelo['mu'][:,0])
pi, mu, S = modelo['pi'][orden], modelo['mu'][orden], modelo['S'][orden]

print(f"{'Perfil':8s} {'peso':>7s} {'dist. típica':>13s} {'dur. típica':>12s} {'velocidad':>10s}")
for k in range(5):
    dist_t, dur_t = np.exp(mu[k,0]), np.exp(mu[k,1])
    vel = dist_t / (dur_t/3600)
    print(f"P{k:<7d} {pi[k]:7.4f} {dist_t:10.2f} km {dur_t/60:9.1f} min {vel:7.1f} km/h")


Perfil      peso  dist. típica  dur. típica  velocidad
P0        0.0280       0.63 km       7.2 min     5.3 km/h
P1        0.4206       1.29 km       7.0 min    11.0 km/h
P2        0.4046       2.78 km      12.7 min    13.1 km/h
P3        0.1285       8.09 km      24.2 min    20.0 km/h
P4        0.0183      20.49 km      44.7 min    27.5 km/h


In [7]:
# Responsabilidades: qué tan segura es la asignación de cada viaje
r, ll = paso_E(X, pi, mu, S)
rmax = r.max(1)
print(f"Responsabilidad máxima: mediana={np.median(rmax):.3f} | >=0.9: {(rmax>=0.9).mean()*100:.2f}% | <0.6 (ambiguos): {(rmax<0.6).mean()*100:.2f}%")


Responsabilidad máxima: mediana=0.754 | >=0.9: 11.74% | <0.6 (ambiguos): 18.19%


### Ejemplo del paso E con un viaje real

In [8]:
i = int(np.argmin(np.abs(rmax - 0.55)))  # un viaje representativo de la zona ambigua
x = X[i]
print(f"Viaje {m.id.iloc[i]}: {m.dist_km.iloc[i]:.2f} km, {m.trip_duration.iloc[i]/60:.1f} min")
lg = np.array([np.log(pi[k]) + log_gauss(x[None,:], mu[k], S[k])[0] for k in range(5)])
for k in range(5): print(f"  P{k}: log(pi)+log N(x) = {lg[k]:9.3f}")
print(f"  log-sum-exp = {logsumexp(lg):.4f}")
print(f"  Responsabilidades = {np.round(np.exp(lg - logsumexp(lg)), 4)}")


Viaje id0710740: 2.25 km, 18.8 min
  P0: log(pi)+log N(x) =    -6.399
  P1: log(pi)+log N(x) =    -2.459
  P2: log(pi)+log N(x) =    -2.229
  P3: log(pi)+log N(x) =    -6.990
  P4: log(pi)+log N(x) = -1023.426
  log-sum-exp = -1.6310
  Responsabilidades = [0.0085 0.4368 0.55   0.0047 0.    ]


In [9]:
# Guardar etiquetas y responsabilidades para la integración (notebook 04)
etiquetas = pd.DataFrame(r, columns=[f'r{k}' for k in range(5)])
etiquetas['id'] = m.id.values
etiquetas['perfil'] = r.argmax(1)
etiquetas['rmax'] = rmax
etiquetas.to_parquet('rol3_etiquetas.parquet', index=False)
print("Guardado: rol3_etiquetas.parquet")


Guardado: rol3_etiquetas.parquet


## Resumen para la integración (Rol 4)

- 5 perfiles: corto lento (P0, posible congestión/GPS), corto urbano (P1, 42%), medio urbano
  (P2, 40%, se solapa con P1), largo interdistrital (P3, 13%), ruta de aeropuerto JFK (P4, 2%,
  distancia casi fija).
- El EM converge al mismo óptimo desde inicializaciones distintas cuando se usa un criterio de
  convergencia estricto — con los valores por defecto de scikit-learn, no.
- 18% de los viajes son ambiguos entre perfiles (responsabilidad máxima < 0.6), sobre todo en
  Manhattan, no en los aeropuertos.